In [13]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

# 1.1 Selección y Prueba de Técnicas

## Rol / Persona

Consiste en decirle al modelo "quién es" para que actue de una forma determinada

**Ejemplo práctico:**
Vamos a decirle al modelo que es un chef y que explique lo que es una api con analogías culinarias

In [14]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": "Actúa como un chef de alta cocina que explica conceptos de software usando analogías culinarias. Explícame qué es una API y cómo funciona en menos de 100 palabras",
        }
    ],
)

print(completion.choices[0].message.content)

Imagina que en un restaurante, la cocina es el backend y el menú es la API. La API es como un camarero que toma tu pedido (solicitud) y se lo lleva a la cocina. Cuando preparan tu plato (respuesta), el camarero lo trae de vuelta a la mesa. Así, tú (el cliente) no necesitas saber cómo se hace la comida, solo interactúas con el menú. La API define cómo se comunican las aplicaciones, permitiendo que diferentes "cocinas" (sistemas) trabajen juntas sin que tengas que entrar en su cocina.


**Análisis**
- ¿Funciona?: Adopta el tono "gastronómico" correctamente
- Ventajas: Elimina la frialdad técnica para explicar este tipo de conceptos y ajusta el vocabulario al nivel del interlocutor
- Casos de uso: Soporte al cliente especializado o simulación de entrevistas

## Especificidad & Constraint / Format forcing

- **Especificidad:**
Consiste en ser muy explícito para obtener una respuesta más ajustada a lo que buscamos

- **Constraint / Format forcing:**
Consiste en forzar al modelo a dar un formato de salida determinado

**Ejemplo Práctico:**
Vamos a decirle al modelo que extraiga información de un texto y con un formato determinado

In [15]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": "Extrae los nombres de empresas y sus tecnologías mencionadas en este texto: 'CloudCorp usa Kubernetes, mientras que DataSafe prefiere Docker y Terraform'. Formato de salida: Una tabla Markdown con dos columnas: Empresa | Tecnología. No añadas introducciones ni conclusiones.",
        }
    ],
)

print(completion.choices[0].message.content)

| Empresa    | Tecnología   |
|------------|--------------|
| CloudCorp  | Kubernetes   |
| DataSafe   | Docker       |
| DataSafe   | Terraform    |


**Análisis**
- ¿Funciona?: Al ser explícito con el formato "Tabla Markdown" y la restricción "sin introducciones", el modelo no "rellenó" con texto innecesario. Funciona correctamente
- Ventajas: Ahorra tiempo de limpieza de datos y reduce el consumo de tokens
- Casos de uso: Automatización de flujos de trabajo que conlleven extracción de datos y generación de informes estructurados

## Few-shot

Consiste en poner ejemplos para influir al modelo con un patrón de respuesta

**Ejemplo Práctico:**
En este caso vamos a extraer información de un mensaje para reservar dandole varios ejemplos para que responda con ese formato.

In [16]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": """Extrae la información de los siguientes mensajes siguiendo estrictamente el formato de los ejemplos.

                        Entrada: 'Reserva para 4 personas, este sábado a las 21:00. Nombre: Juan Pérez.'
                        Salida: {"pax": 4, "dia": "sábado", "hora": "21:00", "cliente": "Juan Pérez"}

                        Entrada: 'Mesa para dos mañana a mediodía (14:30). A nombre de Marta.'
                        Salida: {"pax": 2, "dia": "mañana", "hora": "14:30", "cliente": "Marta"}

                        Entrada: 'Somos un grupo de 10 para el próximo viernes a las 22:00. Apúntalo a nombre de Carlos Ruiz.'
                        Salida:""",
        }
    ],
)

print(completion.choices[0].message.content)

{"pax": 10, "dia": "próximo viernes", "hora": "22:00", "cliente": "Carlos Ruiz"}


**Análisis**
- ¿Funciona?: Sí, el modelo entendió que la información extraida debe ir entre corchetes y que datos tiene que extraer.
- Ventajas: Es la forma más rápida de "entrenar" al modelo para tareas repetitivas sin hacer fine-tuning.
- Casos de uso: Clasificación de correos, etiquetado de bases de datos o traducción con glosarios específicos.

## Chain-of-thought

Consiste en pedirle al modelo que ponga pasos intermedios para que sea más explicativo y poder ver el razonamiento del modelo

**Ejemplo Práctico:**
Vamos a hacer al modelo resolver un problema de lógica/negocio que requiere varios pasos

In [17]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": """Un cliente quiere migrar 15 bases de datos a Azure. Cada migración tarda 3 horas. Tenemos 2 ingenieros trabajando en paralelo. Cada ingeniero trabaja 6 horas al día.
                        Pregunta: ¿Cuántos días tardarán en completar el proyecto?
                        Instrucción: Piensa paso a paso antes de dar la respuesta final.""",
        }
    ],
)

print(completion.choices[0].message.content)

Para calcular cuántos días tardarán en completar la migración de las 15 bases de datos a Azure con los datos proporcionados, sigamos los siguientes pasos:

1. **Determinar cuántas bases de datos pueden migrar en un día:**
   - Cada ingeniero puede trabajar 6 horas al día.
   - Como cada migración tarda 3 horas, cada ingeniero puede migrar:
     \[
     \text{Bases de datos por ingeniero por día} = \frac{6 \text{ horas}}{3 \text{ horas por base de datos}} = 2 \text{ bases de datos}
     \]
   - Como hay 2 ingenieros trabajando en paralelo, el total de bases de datos que pueden migrar en un día es:
     \[
     \text{Total migraciones por día} = 2 \text{ ingenieros} \times 2 \text{ bases de datos por ingeniero} = 4 \text{ bases de datos por día}
     \]

2. **Calcular cuántos días se necesitan para migrar las 15 bases de datos:**
   - Si pueden migrar 4 bases de datos por día, el número total de días necesarios para migrar 15 bases de datos es:
     \[
     \text{Días necesarios} = \frac

**Análisis**
- ¿Funciona?: Sí. Al obligarlo a pensar paso a paso, evitamos que el modelo intente adivinar el número final y fuerce el cálculo intermedio.
- Ventajas: Aumenta la precisión en tareas matemáticas o de razonamiento complejo.
- Casos de uso: Resolución de problemas técnicos, planificación de proyectos o auditoría de procesos.

## Iterative refinement

Consiste en hacer un proceso iterativo pidiendo mejoras continuas hasta llegar al objetivo

**Ejemplo Práctico:**
Vamos a crear un eslogan de una app de gestión de tiempo.

Prompts:
1. Escribe un eslogan para una app de gestión de tiempo.
2. Demasiado aburrido. Hazlo más agresivo, que use una metáfora sobre el dinero y que tenga menos de 5 palabras.
3. Más agresivo, tiene que calar

In [18]:
messages = []

print("--- Inicio del Refinamiento (escribe 'salir' para terminar) ---")

while True:    
    user_input = input("\nUsuario: ")
    
    if user_input.lower() == "salir":
        break

    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"), # Tu gpt-4o-mini
        messages=messages
    )

    ai_response = response.choices[0].message.content
    print(f"\nGPT-4o-mini: {ai_response}")
    
    messages.append({"role": "assistant", "content": ai_response})

--- Inicio del Refinamiento (escribe 'salir' para terminar) ---

GPT-4o-mini: "Domina tu tiempo, transforma tu vida."

GPT-4o-mini: "Convierte minutos en fortuna."

GPT-4o-mini: "Time is cash; gástalo bien."


**Análisis**
- ¿Funciona?: Cumplie con todas las nuevas restricciones y ha mejorado el eslogan.
- Ventajas: Permite llegar a resultados de alta calidad que son difíciles de obtener en un solo intento.
- Casos de uso: Escritura creativa, optimización de código o redacción de correos.

## In-context retrieval

Consiste en darale al modelo contexto o información relevante para que la respuesta se más precisa que sin esa información

**Ejemplo Práctico:**
Vamos a aportarle al modelo información sobre las políticas de teletrabajo en una empresa y vamos a hacerle una pregunta sobre ello.

In [19]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": """CONTEXTO: La política de teletrabajo de 'Empresa X' permite 3 días de oficina y 2 de casa. Los viernes la oficina cierra a las 14:00.
                        PREGUNTA: Según el contexto, ¿puedo trabajar desde casa un viernes?""",
        }
    ],
)

print(completion.choices[0].message.content)

Según la política de teletrabajo de 'Empresa X', tienes 2 días de teletrabajo a la semana. Como los viernes la oficina cierra a las 14:00, puedes optar por trabajar desde casa un viernes, siempre y cuando no superes el límite de 2 días de teletrabajo en la semana. Si ya has utilizado tus 2 días de trabajo desde casa, no podrás trabajar desde casa un viernes.


**Análisis**
- ¿Funciona?: Sí, se mantuvo dentro de los límites del contexto dado sin inventar reglas.
- Ventajas: Reduce las alucinaciones al anclar al modelo a una fuente real.
- Casos de uso: RAG, manuales de usuario o políticas internas.

# 1.2 Aplicación de Técnicas en Casos Reales

## 1º Caso de Uso: Generador de Documentación Técnica

**Situación:** Tienes un fragmento de código complejo (o una nota técnica) y necesitas que la IA genere una documentación en Markdown para tu equipo, basándose solo en lo que le das.

In [20]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": """Actúa como un Technical Writer senior especializado en documentación para desarrolladores.
                        
                        > Código fuente:
                        def calc_retention(users, days): return [u for u in users if u.last_login >= (today - days)]
                        
                        Genera un archivo Markdown con:

                        1. Descripción breve (máx 2 frases).
                        2. Tabla de parámetros (Nombre | Tipo | Descripción).
                        3. Ejemplo de uso. Importante: No inventes funcionalidades que no estén en el código.
                        """,
        }
    ],
)

print(completion.choices[0].message.content)

# Función `calc_retention`

La función `calc_retention` filtra y devuelve una lista de usuarios que han iniciado sesión en los últimos `n` días. Esta herramienta es útil para analizar la retención de usuarios en aplicaciones.

## Parámetros

| Nombre  | Tipo   | Descripción                                                                                 |
|---------|--------|---------------------------------------------------------------------------------------------|
| users   | list   | Lista de objetos de usuario. Cada objeto debe tener un atributo `last_login`.               |
| days    | int    | Número de días que se utilizarán como umbral para filtrar a los usuarios.                 |

## Ejemplo de uso

```python
from datetime import datetime, timedelta

# Supongamos que hoy es 2023-10-05
today = datetime(2023, 10, 5)

# Definimos una clase de usuario para el ejemplo
class User:
    def __init__(self, last_login):
        self.last_login = last_login

# Creamos una lista de usua

**Técnicas Utilizadas:**
- Rol: Se le da un rol al modelo para que sea experto en la solución que se requiere.
- In-context Retrieval: Se le da al modelo el fragemento del código sobre el que tiene que hacer la documentación.
- Constraint / Format forcing: Se le pide al modelo que sea en MD directamente.

## 2º Caso de Uso: Asistente de Datos

**Situación:** Tienes empleados que no saben SQL pero necesitan extraer datos de la base de datos de la empresa. El modelo debe traducir sus peticiones a código SQL válido, optimizado y seguro.

In [21]:
completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": """
                        Actúa como un Arquitecto de Datos experto en SQL Server.
                        
                        > Esquema de tablas:
                        - Clientes (id_cliente, nombre, pais, fecha_registro)
                        - Pedidos (id_pedido, id_cliente, fecha, total, estado)

                        > Entrada: "¿Cuántos clientes tenemos en España?"
                        Salida: SELECT COUNT(*) FROM Clientes WHERE pais = 'España';

                        Antes de escribir el código, explica brevemente qué tablas vas a unir y por qué campos.

                        Devuelve el código SQL dentro de un bloque de código. Usa alias para las tablas para que sea más legible.  

                        Pregunta: Dime el nombre de los 5 clientes que más dinero se han gastado en pedidos completados durante el año 2025.      
                        """,
        }
    ],
)

print(completion.choices[0].message.content)

Para responder a la pregunta sobre los cinco clientes que más dinero han gastado en pedidos completados durante el año 2025, vamos a utilizar las dos tablas: **Clientes** y **Pedidos**. 

Vamos a unir estas dos tablas a través del campo `id_cliente`, que aparece en ambas y permite establecer la relación entre los clientes y sus pedidos. La tabla **Clientes** nos proporcionará información sobre los clientes, mientras que la tabla **Pedidos** nos dará detalles sobre el dinero gastado en los pedidos.

Además, filtraremos los pedidos por el estado "completado" y por las fechas que corresponden al año 2025, y luego sumaremos el total de los pedidos por cliente.

El código SQL para lograr esto sería el siguiente:

```sql
SELECT TOP 5 c.nombre, SUM(p.total) AS total_gastado
FROM Clientes AS c
JOIN Pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.estado = 'completado' 
AND YEAR(p.fecha) = 2025
GROUP BY c.nombre
ORDER BY total_gastado DESC;
``` 

Este código selecciona los nombres de los cli

**Técnicas Utilizadas:**
- Rol: Se le da un rol al modelo para que sea experto en la solución que se requiere.
- In-context Retrieval: Se le da al modelo el esquema de tablas que tiene la base de datos.
- Few-Shot: Se le da al modelo un ejemplo para que sepa que tipo de respuesta tiene que dar.
- Chain-of-thought: Se le pide al modelo que explique los pasos que le llevan a esa conslusión.
- Especificidad: Se le pide al modelo que dé la respuesta en un bloque de código de sql y que use alias para las tablas.

# Conclusiones Personales

Las técnicas que más útiles veo son:

- Rol: Porque ayudamos al modelo a ponerse en contexto de su rol, si no se lo decimos puede interpretar el contexto erróneamente.
- In-context Retrieval: Para aportar información al modelo y reducir las alucinaciones.
- Few-Shot: Para ayudar al modelo a saber que tipo de salida esperamos o que tiene que hacer en determinados casos (razonamientos simples por ej.)
- Chain-of-thought: Para entender al modelo y poder interpretar mejor lo que ha hecho y porque lo ha hecho.
- Format Forcing: Para forzar una salida en un formato concreto, para hacerlo más estricto y tener el formato esperado siempre.